# ⚛️ Kuantum Genlik Tahmini (QAE) ile Kredi Risk Değerlemesi

Bu notebook'ta, **Qiskit Finance** kütüphanesini ve **Iterative Quantum Amplitude Estimation (IQAE)** algoritmasını kullanarak kredi portföyümüzün beklenen kaybını ve Riske Maruz Değerini (Value-at-Risk - VaR) hesaplayacağız.

## 📝 Teorik Altyapı
Kredi risk analizi, portföydeki $N$ adet kredinin temerrüt durumlarını simüle etmeyi hedefler. Her bir kredi için:
*   Temerrüt olasılığı: $p_i$
*   Temerrüt halinde kayıp: $L_i$

Toplam portföy kaybı:
$$L = \sum_{i=1}^N L_i X_i$$
Burada $X_i \in \{0, 1\}$ Bernoulli rastgele değişkenidir ($P(X_i = 1) = p_i$).

Kuantum Genlik Tahmini (QAE), bu kaybın beklenen değerini $\mathbb{E}[L]$ klasik Monte Carlo simülasyonundaki $O(1/\epsilon^2)$ örnekleme karmaşıklığı yerine, **$O(1/\epsilon)$** sorgu karmaşıklığı ile hesaplar. Bu durum NISQ cihazları ve gelecek hata toleranslı kuantum bilgisayarlar için çok büyük bir hızlanma anlamına gelir.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd

# src klasörünü import edebilmek için ekleyelim
sys.path.append(os.path.abspath(os.path.join("..")))
from src.quantum_risk_engine import QuantumCreditRiskEngine

# Bir önceki aşamada ürettiğimiz veriyi yükleyelim
portfolio_path = os.path.join("..", "data", "processed", "predicted_portfolio.csv")
if not os.path.exists(portfolio_path):
    print("Hata: predicted_portfolio.csv bulunamadı! 01_credit_risk_classification notebook'unu veya train_credit_classifier.py'yi çalıştırın.")
else:
    portfolio_df = pd.read_csv(portfolio_path)
    print("Yüklenen Kredi Portföyü:")
    print(portfolio_df)

## 🎲 Klasik Monte Carlo Simülasyonu Baseline'ı

İlk olarak kuantum motorumuzu klasik Monte Carlo simülasyonu ile karşılaştıracağız. Bu aşamada portföyün toplam beklenen kaybını, varyansını ve %95 güven düzeyindeki Riske Maruz Değerini (Value-at-Risk - VaR) hesaplayacağız.

In [ ]:
p = portfolio_df['Default_Probability'].values
l = portfolio_df['Loss_Given_Default'].values

engine = QuantumCreditRiskEngine(probabilities=p, losses=l)

# Monte Carlo Simülasyonu
mc_res = engine.run_monte_carlo(num_samples=50000)
print(f"Klasik Beklenen Kayıp: {mc_res['expected_loss']:.2f} TL")
print(f"Klasik Riske Maruz Değer (VaR %95): {mc_res['var_95']:.2f} TL")
print(f"Klasik Koşullu Riske Maruz Değer (CVaR %95): {mc_res['cvar_95']:.2f} TL")

# Portföy Kayıp Dağılımını Çizdirelim
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.hist(mc_res['portfolio_losses'], bins=20, density=True, alpha=0.6, color='blue', edgecolor='black')
plt.axvline(mc_res['expected_loss'], color='red', linestyle='dashed', linewidth=2, label=f"Beklenen Kayıp: {mc_res['expected_loss']:.2f} TL")
plt.axvline(mc_res['var_95'], color='orange', linestyle='dashed', linewidth=2, label=f"VaR %95: {mc_res['var_95']:.2f} TL")
plt.title("Monte Carlo Portföy Kayıp Dağılımı")
plt.xlabel("Kayıp Tutarı (TL)")
plt.ylabel("Olasılık Yoğunluğu")
plt.legend()
plt.grid(True)
plt.show()

## ⚛️ Kuantum Genlik Tahmini (QAE) Hesaplaması

Kuantum Genlik Tahmini (QAE) motorumuzu çalıştırarak beklenen kaybı kuantum simülasyonu ile hesaplayacağız. `epsilon` parametresi kuantum tahminindeki maksimum hedef hata sınırını belirler.

In [ ]:
# Kuantum QAE motorunu çalıştıralım (epsilon=0.05 hata sınırı ile)
qae_res = engine.run_quantum_qae(epsilon=0.05)

print(f"Kuantum Durumu: {qae_res['quantum_status']}")
print(f"Kuantum Beklenen Kayıp (QAE): {qae_res['expected_loss']:.2f} TL")
print(f"Hedef Hata Payı Sınırı: ±{qae_res['error_bound']*100:.1f}%")
print(f"Kuantum Güven Aralığı (Confidence Interval): [{qae_res['confidence_interval'][0]:.2f}, {qae_res['confidence_interval'][1]:.2f}] TL")

# Karşılaştırma Raporu
print("\n--- KARŞILAŞTIRMA RAPORU ---")
diff_percentage = abs(qae_res['expected_loss'] - mc_res['expected_loss']) / mc_res['expected_loss'] * 100
print(f"Monte Carlo Beklenen Kayıp: {mc_res['expected_loss']:.2f} TL")
print(f"Kuantum QAE Beklenen Kayıp: {qae_res['expected_loss']:.2f} TL")
print(f"Fark Yüzdesi: %{diff_percentage:.2f}")